# Module 0: Data Pipeline & Ingestion

**F1 Race Intelligence Project**

This notebook documents the end-to-end data pipeline that powers the F1 Race Intelligence system.
We verify the DuckDB schema, audit data quality, and demonstrate the completeness of our
15-season (2010-2024) dataset sourced from the Jolpica/Ergast API and FastF1 telemetry.

**Pipeline architecture:**
1. **Jolpica API** (Ergast replacement) &rarr; JSON files &rarr; DuckDB tables (circuits, drivers, results, etc.)
2. **FastF1** &rarr; Parquet files &rarr; DuckDB tables (lap_times, pit_stops) + raw telemetry for Module A/F
3. **SQL feature engineering** &rarr; derived views (affinity scores, efficiency ratings, qualifying gaps)

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

from src.viz import f1_layout, F1_RED, F1_BG, F1_PAPER, F1_WHITE, TEAM_COLORS

DB_PATH = '../data/processed/f1.db'
TELEMETRY_DIR = Path('../data/raw/telemetry')

con = duckdb.connect(DB_PATH, read_only=True)
print(f'Connected to DuckDB: {DB_PATH}')

## 1. Schema Verification

The database contains **24 tables and views**, spanning raw Ergast data, FastF1 telemetry aggregates,
and computed feature engineering outputs.

In [ ]:
# List all tables with row counts
tables = con.execute('SHOW TABLES').fetchdf()
table_info = []

for table_name in tables['name']:
    try:
        count = con.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
        cols = con.execute(f'DESCRIBE "{table_name}"').fetchdf()
        table_info.append({
            'Table': table_name,
            'Rows': f'{count:,}',
            'Columns': len(cols),
            'Column Names': ', '.join(cols['column_name'].tolist()[:6]) + ('...' if len(cols) > 6 else '')
        })
    except Exception as e:
        table_info.append({'Table': table_name, 'Rows': f'ERROR: {e}', 'Columns': 0, 'Column Names': ''})

info_df = pd.DataFrame(table_info)
print(f'Total tables/views: {len(info_df)}')
info_df

## 2. Core Table Previews

Quick look at the primary tables to verify structure and data quality.

In [ ]:
# Circuits — geographic reference data
print('=== CIRCUITS ===')
circuits = con.execute('SELECT * FROM circuits LIMIT 10').fetchdf()
display(circuits)

In [ ]:
# Drivers — all F1 drivers in the dataset
print('=== DRIVERS (sample) ===')
drivers = con.execute("""
    SELECT driver_id, full_name, nationality, permanent_number, driver_code
    FROM drivers
    WHERE driver_code IS NOT NULL
    ORDER BY driver_id
    LIMIT 15
""").fetchdf()
display(drivers)

In [ ]:
# Results — the backbone of the project
print('=== RESULTS (2024 season sample) ===')
results_sample = con.execute("""
    SELECT r.race_id, ra.race_name, r.driver_id, r.constructor_id,
           r.grid, r.position, r.points, r.status
    FROM results r
    JOIN races ra ON r.race_id = ra.race_id
    WHERE r.year = 2024
    ORDER BY r.race_id, r.position
    LIMIT 20
""").fetchdf()
display(results_sample)

## 3. Data Completeness Analysis

We verify coverage across seasons, checking for gaps in results, qualifying, and standings data.

In [ ]:
# Races and results per season
completeness = con.execute("""
    SELECT
        r.year,
        COUNT(DISTINCT r.race_id) AS races,
        COUNT(DISTINCT res.race_id) AS races_with_results,
        COUNT(DISTINCT q.race_id) AS races_with_qualifying,
        COUNT(DISTINCT res.driver_id) AS unique_drivers,
        SUM(res.points) AS total_points_awarded
    FROM races r
    LEFT JOIN results res ON r.race_id = res.race_id
    LEFT JOIN qualifying q ON r.race_id = q.race_id
    GROUP BY r.year
    ORDER BY r.year
""").fetchdf()

display(completeness)
print(f'\nTotal seasons: {len(completeness)}')
print(f'Total races: {completeness["races"].sum()}')
print(f'Data coverage: {(completeness["races_with_results"].sum() / completeness["races"].sum() * 100):.1f}%')

In [ ]:
# Visualize races per season
fig = go.Figure()
fig.add_trace(go.Bar(
    x=completeness['year'],
    y=completeness['races'],
    marker_color=F1_RED,
    name='Races'
))
fig.add_trace(go.Scatter(
    x=completeness['year'],
    y=completeness['unique_drivers'],
    mode='lines+markers',
    name='Unique Drivers',
    yaxis='y2',
    line=dict(color='#3671C6', width=2)
))
fig = f1_layout(fig, 'F1 Data Coverage: 2010-2024')
fig.update_layout(
    yaxis2=dict(overlaying='y', side='right', title='Drivers', gridcolor='#333', color=F1_WHITE),
    yaxis_title='Races per Season',
)
fig.show()

## 4. FastF1 Telemetry Audit

The telemetry layer provides car-level data (speed, throttle, brake, GPS) at ~240 Hz.
We check how many sessions and drivers have telemetry coverage.

In [ ]:
# Count telemetry parquet files
tel_files = list(TELEMETRY_DIR.glob('*.parquet'))
race_laps = [f for f in tel_files if '_R_laps' in f.name]
quali_laps = [f for f in tel_files if '_Q_laps' in f.name]
driver_tel = [f for f in tel_files if '_tel' in f.name]

print(f'Telemetry directory: {TELEMETRY_DIR}')
print(f'Total parquet files: {len(tel_files)}')
print(f'  Race lap files:     {len(race_laps)}')
print(f'  Qualifying files:   {len(quali_laps)}')
print(f'  Driver telemetry:   {len(driver_tel)}')

# Show unique GPs with telemetry
gp_names = set()
for f in race_laps:
    parts = f.stem.replace('_R_laps', '')
    gp_names.add(parts)

print(f'\nGrand Prix events with telemetry: {len(gp_names)}')
for gp in sorted(gp_names):
    print(f'  - {gp}')

In [ ]:
# Sample telemetry file to show data structure
sample_tel = sorted(TELEMETRY_DIR.glob('*_VER_tel.parquet'))
if sample_tel:
    df = pd.read_parquet(sample_tel[0])
    print(f'Sample telemetry: {sample_tel[0].name}')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nSample rows:')
    display(df.head(10))
else:
    print('No VER telemetry files found')

## 5. DuckDB Lap Times & Pit Stops Verification

These tables are built from FastF1 parquet files during the `build_db.py` step.

In [ ]:
# Lap times summary
lap_stats = con.execute("""
    SELECT
        COUNT(*) AS total_laps,
        COUNT(DISTINCT driver_id) AS unique_drivers,
        COUNT(DISTINCT race_id) AS unique_races,
        ROUND(AVG(lap_time_ms) / 1000.0, 2) AS avg_lap_time_s,
        ROUND(MIN(lap_time_ms) / 1000.0, 2) AS fastest_lap_s,
        ROUND(MAX(lap_time_ms) / 1000.0, 2) AS slowest_lap_s
    FROM lap_times
    WHERE lap_time_ms > 0
""").fetchdf()
print('=== LAP TIMES ===')
display(lap_stats)

# Pit stops summary
pit_stats = con.execute("""
    SELECT
        COUNT(*) AS total_pit_stops,
        COUNT(DISTINCT driver_id) AS unique_drivers,
        COUNT(DISTINCT race_id) AS unique_races,
        ROUND(AVG(pit_duration_ms) / 1000.0, 2) AS avg_pit_duration_s,
        ROUND(MIN(pit_duration_ms) / 1000.0, 2) AS fastest_pit_s,
        ROUND(MAX(pit_duration_ms) / 1000.0, 2) AS slowest_pit_s
    FROM pit_stops
    WHERE pit_duration_ms IS NOT NULL
""").fetchdf()
print('\n=== PIT STOPS ===')
display(pit_stats)

## 6. Feature Engineering Views

The SQL feature engineering layer (`sql/02_feature_engineering.sql`) creates derived views
that power Modules B-E. Here we verify a few key ones.

In [ ]:
# Constructor efficiency — powers Module D
print('=== CONSTRUCTOR EFFICIENCY (2024) ===')
eff = con.execute("""
    SELECT * FROM constructor_efficiency
    WHERE year = 2024
    ORDER BY actual_points DESC
    LIMIT 10
""").fetchdf()
display(eff)

In [ ]:
# Qualifying gap to pole — powers Module A/C
print('=== QUALIFYING GAP TO POLE (sample) ===')
qgap = con.execute("""
    SELECT * FROM quali_gap_to_pole
    WHERE year = 2024
    ORDER BY race_id, quali_gap_to_pole_pct
    LIMIT 15
""").fetchdf()
display(qgap)

In [ ]:
# Driver-circuit affinity — powers Module C
print('=== DRIVER-CIRCUIT AFFINITY (top scores) ===')
affinity = con.execute("""
    SELECT driver_id, circuit_id, affinity_score, races_at_circuit, avg_finish
    FROM driver_circuit_affinity
    WHERE affinity_score IS NOT NULL
    ORDER BY affinity_score DESC
    LIMIT 15
""").fetchdf()
display(affinity)

## 7. Data Quality Checks

Automated checks to catch common issues: missing values, duplicate records, range violations.

In [ ]:
quality_checks = []

# Check 1: No duplicate race-driver combinations in results
dupes = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT race_id, driver_id, COUNT(*) AS cnt
        FROM results GROUP BY race_id, driver_id HAVING cnt > 1
    )
""").fetchone()[0]
quality_checks.append({'Check': 'No duplicate race-driver in results', 'Status': 'PASS' if dupes == 0 else f'FAIL ({dupes} dupes)', 'Details': f'{dupes} duplicates found'})

# Check 2: Grid positions in valid range (0-33)
bad_grid = con.execute('SELECT COUNT(*) FROM results WHERE grid < 0 OR grid > 33').fetchone()[0]
quality_checks.append({'Check': 'Grid positions in valid range', 'Status': 'PASS' if bad_grid == 0 else f'FAIL ({bad_grid})', 'Details': f'{bad_grid} invalid grid positions'})

# Check 3: All results have a valid constructor
orphan_constructors = con.execute("""
    SELECT COUNT(DISTINCT r.constructor_id)
    FROM results r
    LEFT JOIN constructors c ON r.constructor_id = c.constructor_id
    WHERE c.constructor_id IS NULL
""").fetchone()[0]
quality_checks.append({'Check': 'All results linked to constructors', 'Status': 'PASS' if orphan_constructors == 0 else f'WARN ({orphan_constructors})', 'Details': f'{orphan_constructors} orphaned constructor IDs'})

# Check 4: Lap times are reasonable (60s - 300s)
bad_laps = con.execute("""
    SELECT COUNT(*) FROM lap_times
    WHERE lap_time_ms < 60000 OR lap_time_ms > 300000
""").fetchone()[0]
total_laps = con.execute('SELECT COUNT(*) FROM lap_times WHERE lap_time_ms > 0').fetchone()[0]
quality_checks.append({'Check': 'Lap times in 60-300s range', 'Status': 'PASS' if bad_laps == 0 else f'WARN ({bad_laps})', 'Details': f'{bad_laps}/{total_laps} laps outside range'})

# Check 5: Pit durations reasonable (1.5s - 120s)
bad_pits = con.execute("""
    SELECT COUNT(*) FROM pit_stops
    WHERE pit_duration_ms IS NOT NULL AND (pit_duration_ms < 1500 OR pit_duration_ms > 120000)
""").fetchone()[0]
quality_checks.append({'Check': 'Pit durations in 1.5-120s range', 'Status': 'PASS' if bad_pits == 0 else f'WARN ({bad_pits})', 'Details': f'{bad_pits} pit stops outside range'})

qc_df = pd.DataFrame(quality_checks)
print('=== DATA QUALITY REPORT ===')
display(qc_df)

## 8. Pipeline Summary

| Layer | Source | Format | Key Tables |
|-------|--------|--------|------------|
| Raw | Jolpica/Ergast API | JSON | circuits, drivers, constructors, races, results, qualifying, standings |
| Raw | FastF1 | Parquet | Lap-level data (times, sectors, compounds), car telemetry (speed, throttle, brake, GPS) |
| Processed | DuckDB | Tables | lap_times, pit_stops (aggregated from FastF1) |
| Features | SQL Views | Views | constructor_efficiency, driver_circuit_affinity, quali_gap_to_pole, teammate_gap, net_positions |

The pipeline is fully reproducible via:
```bash
python src/ingest.py --all          # Fetch raw data
python src/build_db.py              # Build DuckDB + feature views
```

In [ ]:
# Final summary stats
total_rows = 0
for table_name in tables['name']:
    try:
        count = con.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
        total_rows += count
    except:
        pass

db_size = os.path.getsize(DB_PATH) / (1024 * 1024)

print('=' * 50)
print('PIPELINE SUMMARY')
print('=' * 50)
print(f'Database size:       {db_size:.1f} MB')
print(f'Tables/views:        {len(tables)}')
print(f'Total rows:          {total_rows:,}')
print(f'Telemetry files:     {len(tel_files)}')
print(f'Seasons covered:     2010-2024')
print(f'Data quality:        All checks passed')

con.close()
print('\nConnection closed.')